In [ ]:
import os
import sys
from pathlib import Path
import nbformat
from flask import Flask, jsonify, request
from flask_cors import CORS

# Get the project root directory
# When running as notebook, Path.cwd() should be the backend directory
# So project root is parent of cwd
current_dir = Path.cwd()
if current_dir.name == "backend":
    project_root = current_dir.parent
else:
    # Fallback: assume we're in project root
    project_root = current_dir

backend_dir = project_root / "backend"
init_notebook_path = backend_dir / "lib" / "data" / "init.ipynb"

# Load and execute the init notebook
nb = nbformat.read(str(init_notebook_path), as_version=4)
namespace = {}
for cell in nb.cells:
    if cell.cell_type == "code":
        # Replace relative path with absolute path
        code = cell.source
        if "../../../data/data.h5" in code:
            data_path = project_root / "data" / "data.h5"
            # Replace the path string, handling both quoted and unquoted cases
            import re
            # Replace "path" or 'path' with the absolute path
            code = re.sub(r'["\']\.\.\/\.\.\/\.\.\/data\/data\.h5["\']', f'"{str(data_path)}"', code)
        exec(code, namespace)

# Extract functions and variables we need
create_plant = namespace.get('create_plant')
create_empID = namespace.get('create_empID')
hruuid = namespace.get('hruuid')
h5py = namespace.get('h5py')
datetime = namespace.get('datetime')

# Initialize Flask app
# Provide explicit root_path and import_name since we're executing from a notebook
import os
app = Flask('server', root_path=str(backend_dir))
CORS(app, origins=["http://localhost:4321", "http://localhost:3000"])

# HDF5 file path
data_file_path = project_root / "data" / "data.h5"


In [ ]:
def get_h5_file():
    """Open and return the HDF5 file handle"""
    return h5py.File(str(data_file_path), "a")

def create_plant_with_context(f):
    """Create a plant using the provided file handle"""
    # Set up the context that create_plant expects
    namespace['file'] = f
    namespace['plant_group'] = f.require_group("plants")
    # Execute create_plant in the updated namespace
    exec('plant = create_plant()', namespace)
    return namespace['plant']

def create_sequencer_effect(f, effect_type: str, row: int, col: int, properties: dict = None):
    """Create a sequencer effect group in sequencer_effects_properties and return its UUID"""
    effects_props_group = f.require_group("sequencer_effects_properties")
    effect_uuid = hruuid.generate()
    sequencer_effect = effects_props_group.require_group(effect_uuid)
    sequencer_effect.attrs["effect_type"] = effect_type
    sequencer_effect.attrs["sequencer_row"] = row
    sequencer_effect.attrs["sequencer_col"] = col
    sequencer_effect.attrs["timestamp"] = datetime.now().isoformat()
    
    # Store properties as attributes
    if properties:
        for key, value in properties.items():
            sequencer_effect.attrs[f"prop_{key}"] = value
    
    # Return the UUID
    return effect_uuid

def get_sequencer_grid():
    """Read sequencer dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            # Create sequencer if it doesn't exist
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        return grid

def set_sequencer_grid(grid):
    """Write 2x12 grid to sequencer dataset"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]
        else:
            seq[:] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]

def get_effects_grid():
    """Read sequencer_effects dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            # Create sequencer_effects if it doesn't exist
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        return grid

def set_effects_grid(grid):
    """Write 2x12 grid to sequencer_effects dataset"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]
        else:
            seq[:] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]


In [ ]:
@app.route('/api/plants', methods=['POST'])
def create_plant_endpoint():
    """Create a new plant"""
    try:
        f = get_h5_file()
        plant = create_plant_with_context(f)
        plant_id = plant.name.split('/')[-1]  # Get the plant ID from the group name
        timestamp = plant.attrs.get("added_timestamp", "")
        f.close()
        
        return jsonify({
            "id": plant_id,
            "added_timestamp": timestamp
        }), 201
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/plants', methods=['GET'])
def list_plants():
    """List all plants"""
    try:
        plants = []
        with get_h5_file() as f:
            if "plants" in f.keys():
                plant_group = f["plants"]
                for plant_id in plant_group.keys():
                    plant = plant_group[plant_id]
                    plants.append({
                        "id": plant_id,
                        "added_timestamp": plant.attrs.get("added_timestamp", "")
                    })
        return jsonify({"plants": plants}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['GET'])
def get_sequencer():
    """Get current sequencer state"""
    try:
        grid = get_sequencer_grid()
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['PUT'])
def update_sequencer():
    """Update sequencer position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        plant_id = data.get('plant_id', '')  # Empty string to clear
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        grid = get_sequencer_grid()
        grid[row][col] = plant_id
        set_sequencer_grid(grid)
        
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['GET'])
def get_effects():
    """Get current effects grid state - returns effect types by looking up UUIDs from sequencer_effects_properties"""
    try:
        grid = get_effects_grid()
        # Convert UUIDs to effect types for frontend compatibility
        effect_types_grid = []
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            for row in grid:
                effect_types_row = []
                for uuid in row:
                    if uuid:
                        # Look up effect type from sequencer_effects_properties group
                        if uuid in effects_props_group.keys():
                            sequencer_effect = effects_props_group[uuid]
                            effect_type = sequencer_effect.attrs.get("effect_type", "")
                            effect_types_row.append(effect_type)
                        else:
                            # UUID not found, clear it
                            effect_types_row.append("")
                    else:
                        effect_types_row.append("")
                effect_types_grid.append(effect_types_row)
        return jsonify({"effects": effect_types_grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['PUT'])
def update_effects():
    """Update effects grid position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        effect = data.get('effect', '')  # Empty string to clear
        properties = data.get('properties', {})  # Optional properties
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        # Validate effect type
        valid_effects = ['AC', 'DC', 'AMF', 'CMF', '']
        if effect not in valid_effects:
            return jsonify({"error": f"effect must be one of {valid_effects}"}), 400
        
        grid = get_effects_grid()
        
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            
            # If clearing effect, delete the sequencer effect group and clear the grid position
            if not effect:
                old_uuid = grid[row][col]
                if old_uuid and old_uuid in effects_props_group.keys():
                    del effects_props_group[old_uuid]
                grid[row][col] = ""
            else:
                # Check if there's already an effect at this position
                old_uuid = grid[row][col]
                
                # If there's an existing effect, update it instead of creating new
                if old_uuid and old_uuid in effects_props_group.keys():
                    sequencer_effect = effects_props_group[old_uuid]
                    # Update effect type and properties
                    sequencer_effect.attrs["effect_type"] = effect
                    sequencer_effect.attrs["sequencer_row"] = row
                    sequencer_effect.attrs["sequencer_col"] = col
                    sequencer_effect.attrs["timestamp"] = datetime.now().isoformat()
                    
                    # Update properties
                    if properties:
                        for key, value in properties.items():
                            sequencer_effect.attrs[f"prop_{key}"] = value
                    
                    # Keep the same UUID
                    grid[row][col] = old_uuid
                else:
                    # Create new sequencer effect group
                    uuid = create_sequencer_effect(f, effect, row, col, properties)
                    grid[row][col] = uuid
        
        set_effects_grid(grid)
        
        # Return effect types for frontend compatibility
        effect_types_grid = []
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            for grid_row in grid:
                effect_types_row = []
                for uuid in grid_row:
                    if uuid:
                        if uuid in effects_props_group.keys():
                            sequencer_effect = effects_props_group[uuid]
                            effect_type = sequencer_effect.attrs.get("effect_type", "")
                            effect_types_row.append(effect_type)
                        else:
                            effect_types_row.append("")
                    else:
                        effect_types_row.append("")
                effect_types_grid.append(effect_types_row)
        
        return jsonify({"effects": effect_types_grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['GET'])
def get_effect_properties(row, col):
    """Get properties for an effect at a specific position"""
    try:
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        with get_h5_file() as f:
            grid = get_effects_grid()
            uuid = grid[row][col]
            
            if not uuid:
                return jsonify({"properties": {}}), 200
            
            effects_props_group = f.require_group("sequencer_effects_properties")
            if uuid not in effects_props_group.keys():
                return jsonify({"properties": {}}), 200
            
            sequencer_effect = effects_props_group[uuid]
            properties = {}
            
            # Extract all prop_* attributes
            for key in sequencer_effect.attrs.keys():
                if key.startswith("prop_"):
                    prop_name = key[5:]  # Remove "prop_" prefix
                    properties[prop_name] = sequencer_effect.attrs[key]
            
            return jsonify({
                "effect_type": sequencer_effect.attrs.get("effect_type", ""),
                "properties": properties
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['PUT'])
def update_effect_properties(row, col):
    """Update properties for an effect at a specific position"""
    try:
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        data = request.get_json()
        properties = data.get('properties', {})
        
        with get_h5_file() as f:
            grid = get_effects_grid()
            uuid = grid[row][col]
            
            if not uuid:
                return jsonify({"error": "Effect not found at this position"}), 404
            
            effects_props_group = f.require_group("sequencer_effects_properties")
            if uuid not in effects_props_group.keys():
                return jsonify({"error": "Effect not found at this position"}), 404
            
            sequencer_effect = effects_props_group[uuid]
            
            # Update properties
            for key, value in properties.items():
                sequencer_effect.attrs[f"prop_{key}"] = value
            
            # Return updated properties
            updated_properties = {}
            for key in sequencer_effect.attrs.keys():
                if key.startswith("prop_"):
                    prop_name = key[5:]
                    updated_properties[prop_name] = sequencer_effect.attrs[key]
            
            return jsonify({
                "effect_type": sequencer_effect.attrs.get("effect_type", ""),
                "properties": updated_properties
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


In [ ]:
if __name__ == "__main__":
    print(f"Starting Flask server on http://localhost:5000")
    print(f"API endpoints:")
    print(f"  POST /api/plants - Create a new plant")
    print(f"  GET /api/plants - List all plants")
    print(f"  GET /api/sequencer - Get sequencer grid")
    print(f"  PUT /api/sequencer - Update sequencer position")
    print(f"  GET /api/effects - Get effects grid")
    print(f"  PUT /api/effects - Update effects grid position")
    print(f"  GET /api/effects/<row>/<col>/properties - Get effect properties")
    print(f"  PUT /api/effects/<row>/<col>/properties - Update effect properties")
    app.run(host='0.0.0.0', port=5000, debug=True)
